# Transformer-only pipeline for AstronomicalClassification

This notebook trains a strong Transformer model on light-curve sequences only (no CatBoost).


In [ ]:
import os
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve


## Config


In [ ]:
@dataclass
class Config:
    seed: int = 42
    n_folds: int = 5
    max_len: int = 256
    min_obs: int = 20
    batch_size: int = 64
    epochs: int = 20
    patience: int = 3
    lr: float = 3e-4
    weight_decay: float = 1e-4
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 4
    dropout: float = 0.1

CFG = Config()
DATA_ROOT = Path('../datasets/')
FILTERS = ['u', 'g', 'r', 'i', 'z', 'y']
FILTER_TO_IDX = {f: i for i, f in enumerate(FILTERS)}


In [ ]:
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(CFG.seed)


## Data loading and splits


In [ ]:
def load_meta():
    train_log = pd.read_csv(DATA_ROOT / 'train_log.csv')
    test_log = pd.read_csv(DATA_ROOT / 'test_log.csv')
    return train_log, test_log

def get_splits(train_log: pd.DataFrame, n_folds: int = 5):
    splits = sorted(train_log['split'].unique())
    folds = []
    for i in range(n_folds):
        folds.append(splits[i::n_folds])
    return folds

train_log, test_log = load_meta()
folds = get_splits(train_log, n_folds=CFG.n_folds)
print('folds:', folds)


## Light-curve loading


In [ ]:
class Curver:
    def load_lightcurves_for_splits(self, splits: list[str], kind: str) -> pd.DataFrame:
        parts = []
        for split in splits:
            fname = 'train_full_lightcurves.csv' if kind == 'train' else 'test_full_lightcurves.csv'
            path = DATA_ROOT / split / fname
            df = pd.read_csv(path)
            df['split'] = split
            parts.append(df)
        return pd.concat(parts, ignore_index=True)

    def clean_lightcurves(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc = lc[lc['Flux'].notna()]
        lc['Time (MJD)'] = lc['Time (MJD)'].astype(float)
        lc['Flux'] = lc['Flux'].astype(float)
        lc['Flux_err'] = lc['Flux_err'].astype(float)
        return lc

    def add_time_features(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc['t0'] = lc.groupby('object_id')['Time (MJD)'].transform('min')
        lc['dt'] = lc['Time (MJD)'] - lc['t0']
        return lc

curve = Curver()


## Sequence dataset with augmentation


In [ ]:
def filter_objects_by_min_obs(lc: pd.DataFrame, min_obs: int):
    counts = lc.groupby('object_id')['Flux'].count()
    keep_ids = counts[counts >= min_obs].index
    return lc[lc['object_id'].isin(keep_ids)].copy(), set(keep_ids)

def _robust_zscore(x: np.ndarray):
    x = x.astype(float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    scale = mad if mad > 0 else np.nanstd(x)
    if not np.isfinite(scale) or scale == 0:
        return np.zeros_like(x, dtype=float)
    return (x - med) / scale

def build_features(dt, flux, ferr):
    dt = dt.astype(float)
    flux = flux.astype(float)
    ferr = ferr.astype(float)

    dt_norm = dt / (np.max(dt) + 1e-6)
    dt_log = np.log1p(dt)

    flux_z = _robust_zscore(flux)
    snr = np.where(ferr > 0, flux / ferr, 0.0)

    ddt = np.diff(dt, prepend=dt[0])
    dflux = np.diff(flux, prepend=flux[0])

    feats = np.stack([
        dt,
        dt_norm,
        dt_log,
        flux,
        ferr,
        flux_z,
        snr,
        ddt,
        dflux,
    ], axis=1)
    return feats


In [ ]:
class LightcurveSequenceDataset(Dataset):
    def __init__(self, lc: pd.DataFrame, meta: pd.DataFrame, max_len: int, train: bool):
        self.max_len = max_len
        self.train = train
        self.meta = meta.set_index('object_id')
        self.object_ids = []
        self.store = {}

        for obj_id, grp in lc.groupby('object_id'):
            grp = grp.sort_values('dt')
            dt = grp['dt'].to_numpy(dtype=float)
            flux = grp['Flux'].to_numpy(dtype=float)
            ferr = grp['Flux_err'].to_numpy(dtype=float)
            fidx = grp['Filter'].map(FILTER_TO_IDX).fillna(-1).to_numpy(dtype=int)
            self.store[obj_id] = (dt, flux, ferr, fidx)
            self.object_ids.append(obj_id)

        self.labels = self.meta.loc[self.object_ids]['target'].astype(int).to_numpy()

    def __len__(self):
        return len(self.object_ids)

    def __getitem__(self, idx):
        obj_id = self.object_ids[idx]
        dt, flux, ferr, fidx = self.store[obj_id]

        # Random crop for long sequences
        if self.train and len(dt) > self.max_len:
            start = np.random.randint(0, len(dt) - self.max_len + 1)
            end = start + self.max_len
        else:
            start, end = 0, len(dt)

        dt = dt[start:end]
        flux = flux[start:end]
        ferr = ferr[start:end]
        fidx = fidx[start:end]

        # Light augmentation: jitter flux
        if self.train:
            noise = np.random.normal(0.0, 0.01, size=flux.shape)
            flux = flux + noise

        feats = build_features(dt, flux, ferr)
        seq_len = feats.shape[0]

        if seq_len < self.max_len:
            pad = self.max_len - seq_len
            feats = np.pad(feats, ((0, pad), (0, 0)), mode='constant', constant_values=0.0)
            fidx = np.pad(fidx, (0, pad), mode='constant', constant_values=-1)
            mask = np.zeros(self.max_len, dtype=bool)
            mask[:seq_len] = True
        else:
            feats = feats[:self.max_len]
            fidx = fidx[:self.max_len]
            mask = np.ones(self.max_len, dtype=bool)

        y = self.labels[idx]
        return {
            'x': torch.tensor(feats, dtype=torch.float32),
            'filter_idx': torch.tensor(fidx, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
            'y': torch.tensor(y, dtype=torch.float32),
            'object_id': obj_id,
        }


## Model


In [ ]:
class Time2Vec(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.periodic = nn.Linear(1, d_model - 1)

    def forward(self, t):
        t = t.unsqueeze(-1)
        lin = self.linear(t)
        per = torch.sin(self.periodic(t))
        return torch.cat([lin, per], dim=-1)

class AttentionPool(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, h, mask):
        scores = self.score(h).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = torch.sum(h * weights, dim=1)
        return pooled

class TransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_filters: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.filter_emb = nn.Embedding(n_filters + 1, d_model)
        self.input_proj = nn.Linear(input_dim, d_model)
        self.time_enc = Time2Vec(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.pool = AttentionPool(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x, filter_idx, mask):
        dt = x[..., 0]
        x_proj = self.input_proj(x)

        fidx = torch.clamp(filter_idx, min=0, max=self.filter_emb.num_embeddings - 1)
        f_emb = self.filter_emb(fidx)

        t_enc = self.time_enc(dt)
        h = x_proj + f_emb + t_enc

        key_padding_mask = ~mask
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        h = self.norm(h)

        pooled = self.pool(h, mask)
        logits = self.head(pooled).squeeze(-1)
        return logits


## Metrics and helpers


In [ ]:
def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx]), float(f1[idx])

def f1_score_binary(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else 2 * tp / denom


## Train / eval loop


In [ ]:
def make_loaders(train_lc, val_lc, train_meta, val_meta, max_len, batch_size):
    train_ds = LightcurveSequenceDataset(train_lc, train_meta, max_len=max_len, train=True)
    val_ds = LightcurveSequenceDataset(val_lc, val_meta, max_len=max_len, train=False)

    y = train_ds.labels
    class_counts = np.bincount(y)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[y]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, train_ds

def train_one_epoch(model, loader, optimizer, scaler, criterion, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = model(x, fidx, mask)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * y.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_true = []
    for batch in loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)

        logits = model(x, fidx, mask)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)

        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_true.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_true = np.concatenate(all_true)

    thr, best_f1 = best_f1_threshold(all_true, all_probs)
    f1_default = f1_score_binary(all_true, all_probs >= 0.5)

    auc = roc_auc_score(all_true, all_probs) if len(np.unique(all_true)) > 1 else 0.5
    pr_auc = average_precision_score(all_true, all_probs) if len(np.unique(all_true)) > 1 else 0.0

    return {
        'val_loss': total_loss / len(loader.dataset),
        'f1@0.5': f1_default,
        'best_f1': best_f1,
        'best_thr': thr,
        'auc': auc,
        'pr_auc': pr_auc,
    }


## Fold training and OOF


In [ ]:
def prepare_fold_data(train_splits, val_splits):
    train_meta = train_log[train_log['split'].isin(train_splits)].copy()
    val_meta = train_log[train_log['split'].isin(val_splits)].copy()

    train_lc = curve.load_lightcurves_for_splits(train_splits, kind='train')
    val_lc = curve.load_lightcurves_for_splits(val_splits, kind='train')
    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
    val_lc = curve.add_time_features(curve.clean_lightcurves(val_lc))

    train_lc, train_keep = filter_objects_by_min_obs(train_lc, min_obs=CFG.min_obs)
    val_lc, val_keep = filter_objects_by_min_obs(val_lc, min_obs=CFG.min_obs)

    train_meta = train_meta[train_meta['object_id'].isin(train_keep)].copy()
    val_meta = val_meta[val_meta['object_id'].isin(val_keep)].copy()
    return train_lc, val_lc, train_meta, val_meta

def train_fold(fold_id, train_splits, val_splits):
    train_lc, val_lc, train_meta, val_meta = prepare_fold_data(train_splits, val_splits)

    train_loader, val_loader, train_ds = make_loaders(
        train_lc, val_lc, train_meta, val_meta,
        max_len=CFG.max_len, batch_size=CFG.batch_size
    )

    model = TransformerClassifier(
        input_dim=train_ds[0]['x'].shape[-1],
        n_filters=len(FILTERS),
        d_model=CFG.d_model,
        n_heads=CFG.n_heads,
        n_layers=CFG.n_layers,
        dropout=CFG.dropout,
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    pos = (train_ds.labels == 1).sum()
    neg = (train_ds.labels == 0).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_f1 = -1.0
    bad_epochs = 0
    best_path = f'best_transformer_fold{fold_id}.pt'

    for epoch in range(1, CFG.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)

        print(
            f"Fold {fold_id} | Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['val_loss']:.4f} | "
            f"best_f1={val_metrics['best_f1']:.4f} | "
            f"auc={val_metrics['auc']:.4f} | "
            f"pr_auc={val_metrics['pr_auc']:.4f}"
        )

        if val_metrics['best_f1'] > best_f1:
            best_f1 = val_metrics['best_f1']
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
        else:
            bad_epochs += 1

        if bad_epochs >= CFG.patience:
            print('Early stopping.')
            break

        scheduler.step()

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    oof_probs = []
    oof_true = []
    oof_ids = []
    for batch in val_loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        y = batch['y'].to(device)
        logits = model(x, fidx, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        oof_probs.append(probs)
        oof_true.append(y.cpu().numpy())
        oof_ids.extend(batch['object_id'])

    oof_probs = np.concatenate(oof_probs)
    oof_true = np.concatenate(oof_true)
    return oof_probs, oof_true, np.array(oof_ids), best_path


## Run OOF training


In [ ]:
all_splits = sorted(train_log['split'].unique())
folds = get_splits(train_log, n_folds=CFG.n_folds)

oof_pred = np.full(len(train_log), np.nan)
oof_true = train_log['target'].astype(int).to_numpy()
id_to_idx = {oid: i for i, oid in enumerate(train_log['object_id'].values)}

fold_paths = []
for fold_id, val_splits in enumerate(folds):
    train_splits = [s for s in all_splits if s not in val_splits]
    val_probs, val_true, val_ids, best_path = train_fold(fold_id, train_splits, val_splits)
    fold_paths.append(best_path)

    for oid, p in zip(val_ids, val_probs):
        oof_pred[id_to_idx[oid]] = p

mask = np.isfinite(oof_pred)
oof_thr, oof_f1 = best_f1_threshold(oof_true[mask], oof_pred[mask])
print(f'OOF best_f1={oof_f1:.4f}, thr={oof_thr:.2f}')

oof_df = pd.DataFrame({
    'object_id': train_log['object_id'].values,
    'target': oof_true,
    'prob_transformer': oof_pred,
})
oof_df.to_csv('oof_transformer.csv', index=False)
print('Saved oof_transformer.csv')


## Train full model (hold-out metrics) and predict test


In [ ]:
def train_full_with_holdout(holdout_splits):
    train_splits = [s for s in sorted(train_log['split'].unique()) if s not in holdout_splits]

    train_meta = train_log[train_log['split'].isin(train_splits)].copy()
    hold_meta = train_log[train_log['split'].isin(holdout_splits)].copy()

    train_lc = curve.load_lightcurves_for_splits(train_splits, kind='train')
    hold_lc = curve.load_lightcurves_for_splits(holdout_splits, kind='train')
    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
    hold_lc = curve.add_time_features(curve.clean_lightcurves(hold_lc))

    train_lc, train_keep = filter_objects_by_min_obs(train_lc, min_obs=CFG.min_obs)
    hold_lc, hold_keep = filter_objects_by_min_obs(hold_lc, min_obs=CFG.min_obs)
    train_meta = train_meta[train_meta['object_id'].isin(train_keep)].copy()
    hold_meta = hold_meta[hold_meta['object_id'].isin(hold_keep)].copy()

    train_loader, hold_loader, train_ds = make_loaders(
        train_lc, hold_lc, train_meta, hold_meta,
        max_len=CFG.max_len, batch_size=CFG.batch_size
    )

    model = TransformerClassifier(
        input_dim=train_ds[0]['x'].shape[-1],
        n_filters=len(FILTERS),
        d_model=CFG.d_model,
        n_heads=CFG.n_heads,
        n_layers=CFG.n_layers,
        dropout=CFG.dropout,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    pos = (train_ds.labels == 1).sum()
    neg = (train_ds.labels == 0).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_f1 = -1.0
    bad_epochs = 0
    best_path = 'best_transformer_full.pt'

    for epoch in range(1, CFG.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion, device)
        val_metrics = evaluate(model, hold_loader, criterion, device)

        print(
            f"Full | Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['val_loss']:.4f} | "
            f"best_f1={val_metrics['best_f1']:.4f} | "
            f"auc={val_metrics['auc']:.4f} | "
            f"pr_auc={val_metrics['pr_auc']:.4f}"
        )

        if val_metrics['best_f1'] > best_f1:
            best_f1 = val_metrics['best_f1']
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
        else:
            bad_epochs += 1

        if bad_epochs >= CFG.patience:
            print('Early stopping (full).')
            break

        scheduler.step()

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, device, val_metrics

@torch.no_grad()
def predict_test(model, device):
    all_splits = sorted(test_log['split'].unique())
    test_lc = curve.load_lightcurves_for_splits(all_splits, kind='test')
    test_lc = curve.add_time_features(curve.clean_lightcurves(test_lc))

    test_seq_ds = LightcurveSequenceDataset(test_lc, test_log, max_len=CFG.max_len, train=False)
    test_loader = DataLoader(test_seq_ds, batch_size=CFG.batch_size, shuffle=False, drop_last=False)

    model.eval()
    all_probs = []
    all_ids = []
    for batch in test_loader:
        x = batch['x'].to(device)
        fidx = batch['filter_idx'].to(device)
        mask = batch['mask'].to(device)
        logits = model(x, fidx, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_ids.extend(batch['object_id'])

    all_probs = np.concatenate(all_probs)
    return np.array(all_ids), all_probs


In [ ]:
holdout_splits = folds[0]
final_thr = float(oof_thr) if 'oof_thr' in globals() else 0.5

final_model, device, hold_metrics = train_full_with_holdout(holdout_splits)
test_ids, test_probs = predict_test(final_model, device)

test_pred = (test_probs >= final_thr).astype(int)
submission = pd.DataFrame({'object_id': test_ids, 'prediction': test_pred})
submission.to_csv('submission_transformer.csv', index=False)
print('Saved submission_transformer.csv, positives:', int(test_pred.sum()))
